# Matrix Factorization and Leakage Audit - All Families

The fixed split assignments created in Notebooks 2 and 3 are treated as immutable inputs.

For every ligand family and every outer fold:

- `train` entries are observed by the matrix-factorization loss;
- `validation` entries remain in the original interaction matrix but receive mask weight `0`;
- `test` entries remain in the original interaction matrix but receive mask weight `0`;
- validation and test labels are therefore **not replaced by zero** and are **not used by the factorization objective**.


In [1]:
from pathlib import Path
import hashlib
import json
import time
import warnings

import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd()
RAW_DIR = PROJECT_ROOT / "data" / "raw"
SPLIT_DIR = PROJECT_ROOT / "data" / "split"
STATS_DIR = PROJECT_ROOT / "Stats"

STATS_DIR.mkdir(parents=True, exist_ok=True)

print("Project root :", PROJECT_ROOT)
print("Raw folder   :", RAW_DIR)
print("Split folder :", SPLIT_DIR)
print("Stats folder :", STATS_DIR)

assert RAW_DIR.exists(), f"Raw-data folder not found: {RAW_DIR}"
assert SPLIT_DIR.exists(), f"Split folder not found: {SPLIT_DIR}"

Project root : c:\Users\riskf\OneDrive\A-DTI2026
Raw folder   : c:\Users\riskf\OneDrive\A-DTI2026\data\raw
Split folder : c:\Users\riskf\OneDrive\A-DTI2026\data\split
Stats folder : c:\Users\riskf\OneDrive\A-DTI2026\Stats


## 1. Fixed configuration

The matrix orientation used throughout the project is:

```text
Y rows    = proteins / targets
Y columns = compounds / drugs

Sd = protein similarity
St = compound similarity
```

In [2]:
FAMILIES = {
    "enzyme": {
        "display": "Enzyme",
        "prefix": "e",
    },
    "gpcr": {
        "display": "GPCR",
        "prefix": "gpcr",
    },
    "ion_channel": {
        "display": "Ion Channel",
        "prefix": "ic",
    },
    "nuclear_receptor": {
        "display": "Nuclear Receptor",
        "prefix": "nr",
    },
}

SUFFIXES = {
    "Y": "_admat_dgc.txt",
    "St": "_simmat_dc.txt",  # legacy name: compound similarity
    "Sd": "_simmat_dg.txt",  # legacy name: protein similarity
}

N_OUTER_FOLDS = 5

# ---------------------------------------------------------------------
# Controlled-benchmark model parameters
# ---------------------------------------------------------------------

MSCMF_K = 50
MSCMF_GAMMA = 0.1
MSCMF_LAMBDA_PROTEIN = 0.1
MSCMF_LAMBDA_COMPOUND = 0.01
MSCMF_TOL = 1e-3
MSCMF_MAX_ITER = 1000

NNMF_K = 50
NNMF_RANDOM_STATE = 0

# The supplied source leaves sklearn defaults implicit.
# These are made explicit for the masked implementation.
NNMF_MAX_ITER = 200
NNMF_TOL = 1e-4
NNMF_EPS = 1e-10

LMF_K = 50
LMF_LEARNING_RATE = 0.01
LMF_MAX_ITER = 100
LMF_EPS = 1e-12

METHODS = ["MSCMF", "NNMF", "LMF"]

# Version is part of the saved-fit validity check so outputs generated
# by an earlier implementation cannot be resumed accidentally.
IMPLEMENTATION_VERSION = "mf_v2_masked_laplacian"

# Long-running experiment protection.
# Existing outputs are reused only when their saved role-file hash,
# method parameters, and implementation version match the current run.
RESUME_IF_VALID = True

print("Methods:", METHODS)
print("Implementation version:", IMPLEMENTATION_VERSION)

Methods: ['MSCMF', 'NNMF', 'LMF']
Implementation version: mf_v2_masked_laplacian


## 2. Load raw matrices

The raw-file dictionary is explicit so unrelated datasets under `data/raw/` cannot be selected accidentally.

In [3]:
RAW_DATA = {}
INPUT_AUDIT_ROWS = []

for family, cfg in FAMILIES.items():

    files = {
        role: RAW_DIR / f"{cfg['prefix']}{suffix}"
        for role, suffix in SUFFIXES.items()
    }

    for role, path in files.items():
        assert path.exists(), (
            f"{cfg['display']}: missing raw {role} file: {path}"
        )

    Y = pd.read_csv(files["Y"], sep="\t", index_col=0)

    # Legacy file naming:
    #   St -> compound similarity
    #   Sd -> protein similarity
    # Use semantic names from this point forward.
    S_compound = pd.read_csv(
        files["St"],
        sep="\t",
        index_col=0,
    )
    S_protein = pd.read_csv(
        files["Sd"],
        sep="\t",
        index_col=0,
    )

    n_proteins, n_compounds = Y.shape

    assert S_protein.shape == (n_proteins, n_proteins), (
        f"{cfg['display']}: protein similarity does not match Y rows."
    )
    assert S_compound.shape == (n_compounds, n_compounds), (
        f"{cfg['display']}: compound similarity does not match Y columns."
    )

    assert Y.index.is_unique
    assert Y.columns.is_unique
    assert set(np.unique(Y.to_numpy())).issubset({0, 1})

    # Align similarity matrices to the exact Y ordering.
    missing_proteins = set(Y.index) - set(S_protein.index)
    missing_compounds = set(Y.columns) - set(S_compound.index)

    assert not missing_proteins, (
        f"{cfg['display']}: proteins missing from protein similarity matrix."
    )
    assert not missing_compounds, (
        f"{cfg['display']}: compounds missing from compound similarity matrix."
    )

    S_protein = S_protein.loc[Y.index, Y.index]
    S_compound = S_compound.loc[Y.columns, Y.columns]

    RAW_DATA[family] = {
        "Y_df": Y,
        "Y": Y.to_numpy(dtype=np.float64),
        "S_protein": S_protein.to_numpy(dtype=np.float64),
        "S_compound": S_compound.to_numpy(dtype=np.float64),
        "protein_ids": Y.index.astype(str).to_numpy(),
        "compound_ids": Y.columns.astype(str).to_numpy(),
    }

    INPUT_AUDIT_ROWS.append({
        "family": cfg["display"],
        "n_proteins": n_proteins,
        "n_compounds": n_compounds,
        "n_pairs": Y.size,
        "n_positive": int((Y.to_numpy() == 1).sum()),
        "n_non_interaction": int((Y.to_numpy() == 0).sum()),
        "Y_shape": str(Y.shape),
        "protein_similarity_shape": str(S_protein.shape),
        "compound_similarity_shape": str(S_compound.shape),
        "binary_Y": True,
        "orientation": "Y rows=proteins; Y columns=compounds",
    })

    print(
        f"PASS - {cfg['display']}: "
        f"Y={Y.shape}, "
        f"S_protein={S_protein.shape}, "
        f"S_compound={S_compound.shape}"
    )

INPUT_AUDIT = pd.DataFrame(INPUT_AUDIT_ROWS)
display(INPUT_AUDIT)

INPUT_AUDIT.to_excel(
    STATS_DIR / "mf_input_audit.xlsx",
    index=False,
    sheet_name="Input Audit",
)

PASS - Enzyme: Y=(664, 445), S_protein=(664, 664), S_compound=(445, 445)
PASS - GPCR: Y=(95, 223), S_protein=(95, 95), S_compound=(223, 223)
PASS - Ion Channel: Y=(204, 210), S_protein=(204, 204), S_compound=(210, 210)
PASS - Nuclear Receptor: Y=(26, 54), S_protein=(26, 26), S_compound=(54, 54)


,family,n_proteins,n_compounds,n_pairs,n_positive,n_non_interaction,Y_shape,protein_similarity_shape,compound_similarity_shape,binary_Y,orientation
0,Enzyme,664,445,295480,2926,292554,"(664, 445)","(664, 664)","(445, 445)",True,Y rows=proteins; Y columns=compounds
1,GPCR,95,223,21185,635,20550,"(95, 223)","(95, 95)","(223, 223)",True,Y rows=proteins; Y columns=compounds
2,Ion Channel,204,210,42840,1476,41364,"(204, 210)","(204, 204)","(210, 210)",True,Y rows=proteins; Y columns=compounds
3,Nuclear Receptor,26,54,1404,90,1314,"(26, 54)","(26, 26)","(54, 54)",True,Y rows=proteins; Y columns=compounds


## 3. Load Step 06 role assignments and construct the observation masks



In [4]:
def file_sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()


def row_col_columns(df):
    if {"y_row", "y_col"}.issubset(df.columns):
        return "y_row", "y_col"
    if {"protein_index", "compound_index"}.issubset(df.columns):
        return "protein_index", "compound_index"
    raise AssertionError(
        "Role-assignment file must contain either "
        "y_row/y_col or protein_index/compound_index."
    )


ROLE_DATA = {}
MASK_AUDIT_ROWS = []
ROLE_INTEGRITY_ROWS = []

for family, cfg in FAMILIES.items():

    Y = RAW_DATA[family]["Y"]
    n_proteins, n_compounds = Y.shape

    ROLE_DATA[family] = {}

    for outer_fold in range(1, N_OUTER_FOLDS + 1):

        role_file = (
            SPLIT_DIR
            / family
            / f"outer_fold_{outer_fold}_roles.csv"
        )

        assert role_file.exists(), (
            f"{cfg['display']}, fold {outer_fold}: "
            f"missing {role_file}"
        )

        roles = pd.read_csv(role_file)

        assert roles["pair_id"].is_unique
        assert set(roles["role"].unique()) == {
            "train",
            "validation",
            "test",
        }
        assert len(roles) == Y.size

        row_col, col_col = row_col_columns(roles)

        rows = roles[row_col].astype(int).to_numpy()
        cols = roles[col_col].astype(int).to_numpy()

        assert rows.min() >= 0 and rows.max() < n_proteins
        assert cols.min() >= 0 and cols.max() < n_compounds

        # Verify that the role file still corresponds exactly to raw Y.
        y_from_matrix = Y[rows, cols].astype(int)
        y_from_roles = roles["y"].astype(int).to_numpy()

        assert np.array_equal(y_from_matrix, y_from_roles), (
            f"{cfg['display']}, fold {outer_fold}: "
            "role-file labels do not align with raw Y."
        )

        mask = np.zeros_like(Y, dtype=np.float64)

        train_bool = roles["role"].eq("train").to_numpy()
        validation_bool = roles["role"].eq("validation").to_numpy()
        test_bool = roles["role"].eq("test").to_numpy()

        mask[
            rows[train_bool],
            cols[train_bool],
        ] = 1.0

        n_train = int(train_bool.sum())
        n_validation = int(validation_bool.sum())
        n_test = int(test_bool.sum())

        assert int(mask.sum()) == n_train

        assert np.all(
            mask[
                rows[validation_bool],
                cols[validation_bool],
            ] == 0
        )

        assert np.all(
            mask[
                rows[test_bool],
                cols[test_bool],
            ] == 0
        )

        assert np.all(
            mask[
                rows[train_bool],
                cols[train_bool],
            ] == 1
        )

        train_ids = set(roles.loc[train_bool, "pair_id"])
        validation_ids = set(
            roles.loc[validation_bool, "pair_id"]
        )
        test_ids = set(roles.loc[test_bool, "pair_id"])

        train_validation_overlap = len(
            train_ids & validation_ids
        )
        train_test_overlap = len(train_ids & test_ids)
        validation_test_overlap = len(
            validation_ids & test_ids
        )

        assert train_validation_overlap == 0
        assert train_test_overlap == 0
        assert validation_test_overlap == 0

        ROLE_DATA[family][outer_fold] = {
            "roles": roles,
            "mask": mask,
            "role_file": role_file,
            "role_file_sha256": file_sha256(role_file),
            "row_col": row_col,
            "col_col": col_col,
        }

        MASK_AUDIT_ROWS.append({
            "family": cfg["display"],
            "outer_fold": outer_fold,
            "n_pairs": len(roles),
            "n_train": n_train,
            "n_validation": n_validation,
            "n_test": n_test,
            "mask_weight_sum": int(mask.sum()),
            "train_weight_all_one": True,
            "validation_weight_all_zero": True,
            "test_weight_all_zero": True,
            "labels_preserved_in_Y": True,
        })

        ROLE_INTEGRITY_ROWS.append({
            "family": cfg["display"],
            "outer_fold": outer_fold,
            "train_validation_overlap": train_validation_overlap,
            "train_test_overlap": train_test_overlap,
            "validation_test_overlap": validation_test_overlap,
            "all_pair_ids_unique": roles["pair_id"].is_unique,
            "raw_Y_alignment_pass": True,
            "pass": True,
        })

        print(
            f"PASS - {cfg['display']}, fold {outer_fold}: "
            f"train={n_train:,}, "
            f"validation={n_validation:,}, "
            f"test={n_test:,}"
        )

MASK_AUDIT = pd.DataFrame(MASK_AUDIT_ROWS)
ROLE_INTEGRITY_CHECK = pd.DataFrame(
    ROLE_INTEGRITY_ROWS
)

display(MASK_AUDIT)
display(ROLE_INTEGRITY_CHECK)

MASK_AUDIT.to_excel(
    STATS_DIR / "mf_mask_audit.xlsx",
    index=False,
    sheet_name="Mask Audit",
)

ROLE_INTEGRITY_CHECK.to_excel(
    STATS_DIR / "mf_role_integrity_check.xlsx",
    index=False,
    sheet_name="Role Integrity",
)

PASS - Enzyme, fold 1: train=189,107, validation=47,277, test=59,096
PASS - Enzyme, fold 2: train=189,107, validation=47,277, test=59,096
PASS - Enzyme, fold 3: train=189,107, validation=47,277, test=59,096
PASS - Enzyme, fold 4: train=189,107, validation=47,277, test=59,096
PASS - Enzyme, fold 5: train=189,107, validation=47,277, test=59,096
PASS - GPCR, fold 1: train=13,558, validation=3,390, test=4,237
PASS - GPCR, fold 2: train=13,558, validation=3,390, test=4,237
PASS - GPCR, fold 3: train=13,558, validation=3,390, test=4,237
PASS - GPCR, fold 4: train=13,558, validation=3,390, test=4,237
PASS - GPCR, fold 5: train=13,558, validation=3,390, test=4,237
PASS - Ion Channel, fold 1: train=27,417, validation=6,855, test=8,568
PASS - Ion Channel, fold 2: train=27,417, validation=6,855, test=8,568
PASS - Ion Channel, fold 3: train=27,417, validation=6,855, test=8,568
PASS - Ion Channel, fold 4: train=27,417, validation=6,855, test=8,568
PASS - Ion Channel, fold 5: train=27,417, validatio

,family,outer_fold,n_pairs,n_train,n_validation,n_test,mask_weight_sum,train_weight_all_one,validation_weight_all_zero,test_weight_all_zero,labels_preserved_in_Y
0,Enzyme,1,295480,189107,47277,59096,189107,True,True,True,True
1,Enzyme,2,295480,189107,47277,59096,189107,True,True,True,True
2,Enzyme,3,295480,189107,47277,59096,189107,True,True,True,True
3,Enzyme,4,295480,189107,47277,59096,189107,True,True,True,True
4,Enzyme,5,295480,189107,47277,59096,189107,True,True,True,True
5,GPCR,1,21185,13558,3390,4237,13558,True,True,True,True
6,GPCR,2,21185,13558,3390,4237,13558,True,True,True,True
7,GPCR,3,21185,13558,3390,4237,13558,True,True,True,True
8,GPCR,4,21185,13558,3390,4237,13558,True,True,True,True
9,GPCR,5,21185,13558,3390,4237,13558,True,True,True,True


,family,outer_fold,train_validation_overlap,train_test_overlap,validation_test_overlap,all_pair_ids_unique,raw_Y_alignment_pass,pass
0,Enzyme,1,0,0,0,True,True,True
1,Enzyme,2,0,0,0,True,True,True
2,Enzyme,3,0,0,0,True,True,True
3,Enzyme,4,0,0,0,True,True,True
4,Enzyme,5,0,0,0,True,True,True
5,GPCR,1,0,0,0,True,True,True
6,GPCR,2,0,0,0,True,True,True
7,GPCR,3,0,0,0,True,True,True
8,GPCR,4,0,0,0,True,True,True
9,GPCR,5,0,0,0,True,True,True


## Step 08 pre-fit leakage audit: held-out-label perturbation invariance

In [5]:
LEAKAGE_PRECHECK_ROWS = []

for family, cfg in FAMILIES.items():

    Y = RAW_DATA[family]["Y"]

    for outer_fold in range(1, N_OUTER_FOLDS + 1):

        entry = ROLE_DATA[family][outer_fold]
        roles = entry["roles"]
        mask = entry["mask"]

        row_col = entry["row_col"]
        col_col = entry["col_col"]

        rows = roles[row_col].astype(int).to_numpy()
        cols = roles[col_col].astype(int).to_numpy()

        heldout = roles["role"].isin(
            ["validation", "test"]
        ).to_numpy()

        Y_perturbed = Y.copy()

        # Temporary audit only. Raw Y is never modified.
        heldout_rows = rows[heldout]
        heldout_cols = cols[heldout]

        Y_perturbed[
            heldout_rows,
            heldout_cols,
        ] = 1.0 - Y_perturbed[
            heldout_rows,
            heldout_cols,
        ]

        masked_original = mask * Y
        masked_perturbed = mask * Y_perturbed

        invariant = np.array_equal(
            masked_original,
            masked_perturbed,
        )

        assert invariant, (
            f"{cfg['display']}, fold {outer_fold}: "
            "held-out label perturbation changed the masked target."
        )

        LEAKAGE_PRECHECK_ROWS.append({
            "family": cfg["display"],
            "outer_fold": outer_fold,
            "n_heldout_labels_perturbed": int(
                heldout.sum()
            ),
            "masked_target_identical_after_perturbation": invariant,
            "pass": invariant,
        })

        print(
            f"PASS - leakage precheck - "
            f"{cfg['display']}, fold {outer_fold}"
        )

LEAKAGE_PRECHECK = pd.DataFrame(
    LEAKAGE_PRECHECK_ROWS
)

display(LEAKAGE_PRECHECK)

LEAKAGE_PRECHECK.to_excel(
    STATS_DIR / "mf_leakage_precheck.xlsx",
    index=False,
    sheet_name="Leakage Precheck",
)

PASS - leakage precheck - Enzyme, fold 1
PASS - leakage precheck - Enzyme, fold 2
PASS - leakage precheck - Enzyme, fold 3
PASS - leakage precheck - Enzyme, fold 4
PASS - leakage precheck - Enzyme, fold 5
PASS - leakage precheck - GPCR, fold 1
PASS - leakage precheck - GPCR, fold 2
PASS - leakage precheck - GPCR, fold 3
PASS - leakage precheck - GPCR, fold 4
PASS - leakage precheck - GPCR, fold 5
PASS - leakage precheck - Ion Channel, fold 1
PASS - leakage precheck - Ion Channel, fold 2
PASS - leakage precheck - Ion Channel, fold 3
PASS - leakage precheck - Ion Channel, fold 4
PASS - leakage precheck - Ion Channel, fold 5
PASS - leakage precheck - Nuclear Receptor, fold 1
PASS - leakage precheck - Nuclear Receptor, fold 2
PASS - leakage precheck - Nuclear Receptor, fold 3
PASS - leakage precheck - Nuclear Receptor, fold 4
PASS - leakage precheck - Nuclear Receptor, fold 5


,family,outer_fold,n_heldout_labels_perturbed,masked_target_identical_after_perturbation,pass
0,Enzyme,1,106373,True,True
1,Enzyme,2,106373,True,True
2,Enzyme,3,106373,True,True
3,Enzyme,4,106373,True,True
4,Enzyme,5,106373,True,True
5,GPCR,1,7627,True,True
6,GPCR,2,7627,True,True
7,GPCR,3,7627,True,True
8,GPCR,4,7627,True,True
9,GPCR,5,7627,True,True


# Step 07 — Matrix-factorization implementations

## 5. MSCMF — masked graph-Laplacian formulation



In [6]:
def graph_laplacian(S):
    degree = S.sum(axis=1)
    L = np.diag(degree) - S
    return L, degree


def mscmf_objective(
    Y,
    A,
    B,
    mask,
    L_protein,
    L_compound,
    gamma,
    lambda_protein,
    lambda_compound,
):
    diff_matrix = Y - A.dot(B.T)
    weighted_diff = mask * diff_matrix

    reconstruction_error = (
        np.linalg.norm(weighted_diff, "fro") ** 2
    )

    protein_graph_reg = (
        lambda_protein
        * np.trace(
            A.T @ L_protein @ A
        )
    )

    compound_graph_reg = (
        lambda_compound
        * np.trace(
            B.T @ L_compound @ B
        )
    )

    latent_reg = gamma * (
        np.linalg.norm(A, "fro") ** 2
        + np.linalg.norm(B, "fro") ** 2
    )

    return (
        reconstruction_error
        + protein_graph_reg
        + compound_graph_reg
        + latent_reg
    )


def mscmf_update_A(
    Y,
    A,
    B,
    mask,
    S_protein,
    protein_degree,
    gamma,
    lambda_protein,
    K,
):
    m, _ = Y.shape
    A_new = np.zeros_like(A)
    identity = np.eye(K)

    for i in range(m):

        observed = mask[i, :] > 0

        B_obs = B[observed, :]
        y_obs = Y[i, observed]

        neighbor_term = (
            lambda_protein
            * (
                S_protein[i, :]
                @ A
            )
        )

        part1 = (
            B_obs.T @ y_obs
            + neighbor_term
        )

        part2 = (
            B_obs.T @ B_obs
            + (
                gamma
                + lambda_protein
                * protein_degree[i]
            )
            * identity
        )

        A_new[i, :] = np.linalg.solve(
            part2,
            part1,
        )

    return A_new


def mscmf_update_B(
    Y,
    A,
    B,
    mask,
    S_compound,
    compound_degree,
    gamma,
    lambda_compound,
    K,
):
    _, n = Y.shape
    B_new = np.zeros_like(B)
    identity = np.eye(K)

    for j in range(n):

        observed = mask[:, j] > 0

        A_obs = A[observed, :]
        y_obs = Y[observed, j]

        neighbor_term = (
            lambda_compound
            * (
                S_compound[j, :]
                @ B
            )
        )

        part1 = (
            A_obs.T @ y_obs
            + neighbor_term
        )

        part2 = (
            A_obs.T @ A_obs
            + (
                gamma
                + lambda_compound
                * compound_degree[j]
            )
            * identity
        )

        B_new[j, :] = np.linalg.solve(
            part2,
            part1,
        )

    return B_new


def fit_mscmf(
    Y,
    mask,
    S_protein,
    S_compound,
    K=MSCMF_K,
    gamma=MSCMF_GAMMA,
    lambda_protein=MSCMF_LAMBDA_PROTEIN,
    lambda_compound=MSCMF_LAMBDA_COMPOUND,
    tolerance=MSCMF_TOL,
    max_iterations=MSCMF_MAX_ITER,
):

    m, n = Y.shape

    L_protein, protein_degree = (
        graph_laplacian(S_protein)
    )
    L_compound, compound_degree = (
        graph_laplacian(S_compound)
    )

    # Source notebook used np.random.rand without a fixed seed.
    A = np.random.rand(m, K)
    B = np.random.rand(n, K)

    objective_history = [
        mscmf_objective(
            Y,
            A,
            B,
            mask,
            L_protein,
            L_compound,
            gamma,
            lambda_protein,
            lambda_compound,
        )
    ]

    converged = False

    for iteration in range(max_iterations):

        A = mscmf_update_A(
            Y,
            A,
            B,
            mask,
            S_protein,
            protein_degree,
            gamma,
            lambda_protein,
            K,
        )

        B = mscmf_update_B(
            Y,
            A,
            B,
            mask,
            S_compound,
            compound_degree,
            gamma,
            lambda_compound,
            K,
        )

        current_objective = mscmf_objective(
            Y,
            A,
            B,
            mask,
            L_protein,
            L_compound,
            gamma,
            lambda_protein,
            lambda_compound,
        )

        objective_history.append(
            current_objective
        )

        difference = abs(
            objective_history[-1]
            - objective_history[-2]
        )

        if difference < tolerance:
            converged = True
            break

    return {
        "protein_embeddings": A,
        "compound_embeddings": B,
        "iterations": iteration + 1,
        "converged": converged,
        "final_objective": float(
            objective_history[-1]
        ),
        "objective_history": objective_history,
    }

## 6. NNMF — masked non-negative factorization



In [7]:
def masked_nnmf_error(Y, A, B, mask):
    residual = mask * (Y - A @ B.T)
    return float(
        np.linalg.norm(residual, "fro")
    )


def fit_masked_nnmf(
    Y,
    mask,
    K=NNMF_K,
    random_state=NNMF_RANDOM_STATE,
    max_iterations=NNMF_MAX_ITER,
    tolerance=NNMF_TOL,
    eps=NNMF_EPS,
):

    m, n = Y.shape

    rng = np.random.RandomState(
        random_state
    )

    # Positive random initialization consistent with
    # the source model's init='random'.
    A = rng.random_sample((m, K))
    B = rng.random_sample((n, K))

    error_history = []
    converged = False

    masked_Y = mask * Y

    previous_error = None

    for iteration in range(max_iterations):

        reconstruction = A @ B.T

        numerator_A = masked_Y @ B
        denominator_A = (
            (mask * reconstruction) @ B
            + eps
        )

        A *= numerator_A / denominator_A
        A = np.maximum(A, eps)

        reconstruction = A @ B.T

        numerator_B = masked_Y.T @ A
        denominator_B = (
            (mask * reconstruction).T @ A
            + eps
        )

        B *= numerator_B / denominator_B
        B = np.maximum(B, eps)

        current_error = masked_nnmf_error(
            Y,
            A,
            B,
            mask,
        )

        error_history.append(current_error)

        if previous_error is not None:

            relative_change = abs(
                previous_error - current_error
            ) / max(abs(previous_error), eps)

            if relative_change < tolerance:
                converged = True
                break

        previous_error = current_error

    return {
        "protein_embeddings": A,
        "compound_embeddings": B,
        "iterations": iteration + 1,
        "converged": converged,
        "final_objective": float(
            error_history[-1]
        ),
        "objective_history": error_history,
    }

## 7. LMF — masked adaptation of the supplied logistic factorization



In [8]:
def sigmoid(x):
    # Numerical safeguard; it does not expose held-out labels.
    x = np.clip(x, -50, 50)
    return 1.0 / (1.0 + np.exp(-x))


def masked_logistic_loss_and_grad(
    A,
    B,
    Y,
    mask,
    eps=LMF_EPS,
):
    predictions = sigmoid(A.dot(B.T))

    observed = mask.astype(bool)

    y_obs = Y[observed]
    p_obs = np.clip(
        predictions[observed],
        eps,
        1.0 - eps,
    )

    loss = -np.sum(
        y_obs * np.log(p_obs)
        + (1.0 - y_obs) * np.log(1.0 - p_obs)
    )

    error = (
        predictions - Y
    ) * mask

    grad_A = error.dot(B)
    grad_B = error.T.dot(A)

    return (
        float(loss),
        grad_A,
        grad_B,
    )


def fit_lmf(
    Y,
    mask,
    K=LMF_K,
    learning_rate=LMF_LEARNING_RATE,
    max_iterations=LMF_MAX_ITER,
):

    # Source notebook used np.random.rand without a fixed seed.
    A = (
        np.random.rand(Y.shape[0], K)
        * 0.01
    )
    B = (
        np.random.rand(Y.shape[1], K)
        * 0.01
    )

    loss_history = []

    for iteration in range(max_iterations):

        loss, grad_A, grad_B = (
            masked_logistic_loss_and_grad(
                A,
                B,
                Y,
                mask,
            )
        )

        A -= learning_rate * grad_A
        B -= learning_rate * grad_B

        loss_history.append(loss)

    return {
        "protein_embeddings": A,
        "compound_embeddings": B,
        "iterations": max_iterations,
        "converged": None,
        "final_objective": float(
            loss_history[-1]
        ),
        "objective_history": loss_history,
    }

## 8. Shared fit, output, and audit utilities



In [9]:
def method_parameters(method):
    if method == "MSCMF":
        return {
            "K": MSCMF_K,
            "gamma": MSCMF_GAMMA,
            "lambda_protein": MSCMF_LAMBDA_PROTEIN,
            "lambda_compound": MSCMF_LAMBDA_COMPOUND,
            "tolerance": MSCMF_TOL,
            "max_iterations": MSCMF_MAX_ITER,
            "initialization_seed": None,
        }

    if method == "NNMF":
        return {
            "K": NNMF_K,
            "random_state": NNMF_RANDOM_STATE,
            "max_iterations": NNMF_MAX_ITER,
            "tolerance": NNMF_TOL,
        }

    if method == "LMF":
        return {
            "K": LMF_K,
            "learning_rate": LMF_LEARNING_RATE,
            "max_iterations": LMF_MAX_ITER,
            "l2_regularization": False,
            "initialization_seed": None,
        }

    raise ValueError(method)


def fit_method(
    method,
    Y,
    mask,
    S_protein,
    S_compound,
):
    if method == "MSCMF":
        return fit_mscmf(
            Y,
            mask,
            S_protein,
            S_compound,
        )

    if method == "NNMF":
        return fit_masked_nnmf(
            Y,
            mask,
        )

    if method == "LMF":
        return fit_lmf(
            Y,
            mask,
        )

    raise ValueError(method)


def fit_output_dir(
    family,
    outer_fold,
    method,
):
    return (
        SPLIT_DIR
        / family
        / "latent"
        / f"outer_fold_{outer_fold}"
        / method.lower()
    )


def expected_metadata(
    family,
    outer_fold,
    method,
):
    entry = ROLE_DATA[family][outer_fold]

    return {
        "implementation_version": IMPLEMENTATION_VERSION,
        "family": family,
        "outer_fold": outer_fold,
        "method": method,
        "role_file_sha256": (
            entry["role_file_sha256"]
        ),
        "parameters": method_parameters(
            method
        ),
    }


def can_resume(
    family,
    outer_fold,
    method,
):
    output_dir = fit_output_dir(
        family,
        outer_fold,
        method,
    )

    protein_file = (
        output_dir
        / "protein_embeddings.npy"
    )
    compound_file = (
        output_dir
        / "compound_embeddings.npy"
    )
    metadata_file = (
        output_dir
        / "fit_metadata.json"
    )

    if not (
        protein_file.exists()
        and compound_file.exists()
        and metadata_file.exists()
    ):
        return False

    try:
        saved_metadata = json.loads(
            metadata_file.read_text(
                encoding="utf-8"
            )
        )
    except Exception:
        return False

    return (
        saved_metadata.get(
            "implementation_version"
        )
        == IMPLEMENTATION_VERSION
        and saved_metadata.get("family")
        == family
        and saved_metadata.get("outer_fold")
        == outer_fold
        and saved_metadata.get("method")
        == method
        and saved_metadata.get(
            "role_file_sha256"
        )
        == ROLE_DATA[family][outer_fold][
            "role_file_sha256"
        ]
        and saved_metadata.get("parameters")
        == method_parameters(method)
    )


def save_fit(
    family,
    outer_fold,
    method,
    result,
    elapsed_seconds,
):
    output_dir = fit_output_dir(
        family,
        outer_fold,
        method,
    )
    output_dir.mkdir(
        parents=True,
        exist_ok=True,
    )

    A = result["protein_embeddings"]
    B = result["compound_embeddings"]

    np.save(
        output_dir
        / "protein_embeddings.npy",
        A.astype(np.float32),
    )

    np.save(
        output_dir
        / "compound_embeddings.npy",
        B.astype(np.float32),
    )

    np.save(
        output_dir
        / "objective_history.npy",
        np.asarray(
            result["objective_history"],
            dtype=np.float64,
        ),
    )

    metadata = expected_metadata(
        family,
        outer_fold,
        method,
    )

    metadata.update({
        "protein_embedding_shape": list(
            A.shape
        ),
        "compound_embedding_shape": list(
            B.shape
        ),
        "iterations": int(
            result["iterations"]
        ),
        "converged": result["converged"],
        "final_objective": float(
            result["final_objective"]
        ),
        "elapsed_seconds": float(
            elapsed_seconds
        ),
    })

    (
        output_dir
        / "fit_metadata.json"
    ).write_text(
        json.dumps(
            metadata,
            indent=2,
        ),
        encoding="utf-8",
    )


def load_saved_fit(
    family,
    outer_fold,
    method,
):
    output_dir = fit_output_dir(
        family,
        outer_fold,
        method,
    )

    A = np.load(
        output_dir
        / "protein_embeddings.npy"
    ).astype(np.float64)

    B = np.load(
        output_dir
        / "compound_embeddings.npy"
    ).astype(np.float64)

    history = np.load(
        output_dir
        / "objective_history.npy"
    ).astype(np.float64)

    metadata = json.loads(
        (
            output_dir
            / "fit_metadata.json"
        ).read_text(
            encoding="utf-8"
        )
    )

    return {
        "protein_embeddings": A,
        "compound_embeddings": B,
        "iterations": metadata[
            "iterations"
        ],
        "converged": metadata[
            "converged"
        ],
        "final_objective": metadata[
            "final_objective"
        ],
        "objective_history": history.tolist(),
        "elapsed_seconds": metadata[
            "elapsed_seconds"
        ],
    }

## 9. Save protein and compound ID order



In [10]:
for family, cfg in FAMILIES.items():

    latent_root = (
        SPLIT_DIR
        / family
        / "latent"
    )
    latent_root.mkdir(
        parents=True,
        exist_ok=True,
    )

    pd.DataFrame({
        "protein_index": np.arange(
            len(
                RAW_DATA[family][
                    "protein_ids"
                ]
            )
        ),
        "protein_id": RAW_DATA[family][
            "protein_ids"
        ],
    }).to_csv(
        latent_root / "protein_ids.csv",
        index=False,
    )

    pd.DataFrame({
        "compound_index": np.arange(
            len(
                RAW_DATA[family][
                    "compound_ids"
                ]
            )
        ),
        "compound_id": RAW_DATA[family][
            "compound_ids"
        ],
    }).to_csv(
        latent_root / "compound_ids.csv",
        index=False,
    )

    print(
        f"Saved ID order - "
        f"{cfg['display']}"
    )

Saved ID order - Enzyme
Saved ID order - GPCR
Saved ID order - Ion Channel
Saved ID order - Nuclear Receptor


## 10. Fit all family × outer-fold × method combinations



In [11]:
FIT_STATS_ROWS = []
EMBEDDING_AUDIT_ROWS = []

for family, cfg in FAMILIES.items():

    Y = RAW_DATA[family]["Y"]
    S_protein = RAW_DATA[family]["S_protein"]
    S_compound = RAW_DATA[family]["S_compound"]

    for outer_fold in range(
        1,
        N_OUTER_FOLDS + 1,
    ):

        mask = ROLE_DATA[
            family
        ][outer_fold]["mask"]

        for method in METHODS:

            print(
                "\n"
                + "=" * 80
            )
            print(
                f"{cfg['display']} | "
                f"outer fold {outer_fold} | "
                f"{method}"
            )
            print(
                "=" * 80
            )

            resumed = False

            if (
                RESUME_IF_VALID
                and can_resume(
                    family,
                    outer_fold,
                    method,
                )
            ):
                print(
                    "Valid saved fit found. "
                    "Loading existing embeddings."
                )

                result = load_saved_fit(
                    family,
                    outer_fold,
                    method,
                )
                elapsed_seconds = result[
                    "elapsed_seconds"
                ]
                resumed = True

            else:
                start = time.time()

                result = fit_method(
                    method,
                    Y,
                    mask,
                    S_protein,
                    S_compound,
                )

                elapsed_seconds = (
                    time.time() - start
                )

                save_fit(
                    family,
                    outer_fold,
                    method,
                    result,
                    elapsed_seconds,
                )

            A = result[
                "protein_embeddings"
            ]
            B = result[
                "compound_embeddings"
            ]

            expected_k = (
                MSCMF_K
                if method == "MSCMF"
                else NNMF_K
                if method == "NNMF"
                else LMF_K
            )

            shape_pass = (
                A.shape
                == (
                    Y.shape[0],
                    expected_k,
                )
                and B.shape
                == (
                    Y.shape[1],
                    expected_k,
                )
            )

            finite_pass = (
                np.isfinite(A).all()
                and np.isfinite(B).all()
            )

            if method == "NNMF":
                nonnegative_pass = (
                    (A >= 0).all()
                    and (B >= 0).all()
                )
            else:
                nonnegative_pass = None

            assert shape_pass, (
                f"{cfg['display']}, "
                f"fold {outer_fold}, "
                f"{method}: invalid "
                "embedding shape."
            )

            assert finite_pass, (
                f"{cfg['display']}, "
                f"fold {outer_fold}, "
                f"{method}: NaN/Inf "
                "detected."
            )

            if method == "NNMF":
                assert nonnegative_pass, (
                    f"{cfg['display']}, "
                    f"fold {outer_fold}: "
                    "NNMF contains negative "
                    "latent values."
                )

            FIT_STATS_ROWS.append({
                "family": cfg["display"],
                "outer_fold": outer_fold,
                "method": method,
                "latent_dimension": expected_k,
                "pair_feature_dimension": (
                    2 * expected_k
                ),
                "n_train_entries": int(
                    mask.sum()
                ),
                "n_masked_entries": int(
                    mask.size - mask.sum()
                ),
                "iterations": result[
                    "iterations"
                ],
                "converged": result[
                    "converged"
                ],
                "final_objective": result[
                    "final_objective"
                ],
                "elapsed_seconds": (
                    elapsed_seconds
                ),
                "resumed_existing_fit": (
                    resumed
                ),
            })

            EMBEDDING_AUDIT_ROWS.append({
                "family": cfg["display"],
                "outer_fold": outer_fold,
                "method": method,
                "protein_embedding_shape": (
                    str(A.shape)
                ),
                "compound_embedding_shape": (
                    str(B.shape)
                ),
                "expected_latent_dimension": (
                    expected_k
                ),
                "shape_pass": shape_pass,
                "finite_values_pass": (
                    finite_pass
                ),
                "nonnegative_pass": (
                    nonnegative_pass
                ),
                "pass": (
                    shape_pass
                    and finite_pass
                    and (
                        method != "NNMF"
                        or nonnegative_pass
                    )
                ),
            })

            # Save progress after each fit.
            FIT_STATS = pd.DataFrame(
                FIT_STATS_ROWS
            )
            EMBEDDING_AUDIT = (
                pd.DataFrame(
                    EMBEDDING_AUDIT_ROWS
                )
            )

            FIT_STATS.to_excel(
                STATS_DIR
                / "mf_fit_statistics.xlsx",
                index=False,
                sheet_name="Fit Statistics",
            )

            EMBEDDING_AUDIT.to_excel(
                STATS_DIR
                / "mf_embedding_audit.xlsx",
                index=False,
                sheet_name="Embedding Audit",
            )

            print(
                f"PASS - {cfg['display']} | "
                f"fold {outer_fold} | "
                f"{method}"
            )

FIT_STATS = pd.DataFrame(
    FIT_STATS_ROWS
)
EMBEDDING_AUDIT = pd.DataFrame(
    EMBEDDING_AUDIT_ROWS
)

display(FIT_STATS)
display(EMBEDDING_AUDIT)


Enzyme | outer fold 1 | MSCMF
PASS - Enzyme | fold 1 | MSCMF

Enzyme | outer fold 1 | NNMF
PASS - Enzyme | fold 1 | NNMF

Enzyme | outer fold 1 | LMF
PASS - Enzyme | fold 1 | LMF

Enzyme | outer fold 2 | MSCMF
PASS - Enzyme | fold 2 | MSCMF

Enzyme | outer fold 2 | NNMF
PASS - Enzyme | fold 2 | NNMF

Enzyme | outer fold 2 | LMF
PASS - Enzyme | fold 2 | LMF

Enzyme | outer fold 3 | MSCMF
PASS - Enzyme | fold 3 | MSCMF

Enzyme | outer fold 3 | NNMF
PASS - Enzyme | fold 3 | NNMF

Enzyme | outer fold 3 | LMF
PASS - Enzyme | fold 3 | LMF

Enzyme | outer fold 4 | MSCMF
PASS - Enzyme | fold 4 | MSCMF

Enzyme | outer fold 4 | NNMF
PASS - Enzyme | fold 4 | NNMF

Enzyme | outer fold 4 | LMF
PASS - Enzyme | fold 4 | LMF

Enzyme | outer fold 5 | MSCMF
PASS - Enzyme | fold 5 | MSCMF

Enzyme | outer fold 5 | NNMF
PASS - Enzyme | fold 5 | NNMF

Enzyme | outer fold 5 | LMF
PASS - Enzyme | fold 5 | LMF

GPCR | outer fold 1 | MSCMF
PASS - GPCR | fold 1 | MSCMF

GPCR | outer fold 1 | NNMF
PASS - GPCR | 

,family,outer_fold,method,latent_dimension,pair_feature_dimension,n_train_entries,n_masked_entries,iterations,converged,final_objective,elapsed_seconds,resumed_existing_fit
0,Enzyme,1,MSCMF,50,100,189107,106373,155,True,592.782721,13.929893,False
1,Enzyme,1,NNMF,50,100,189107,106373,200,False,10.139296,1.017149,False
2,Enzyme,1,LMF,50,100,189107,106373,100,None,2801.031945,1.107309,False
3,Enzyme,2,MSCMF,50,100,189107,106373,101,True,577.340500,10.034239,False
4,Enzyme,2,NNMF,50,100,189107,106373,179,True,10.011409,0.904687,False
5,Enzyme,2,LMF,50,100,189107,106373,100,None,2695.358295,1.064323,False
6,Enzyme,3,MSCMF,50,100,189107,106373,101,True,585.872432,9.604104,False
7,Enzyme,3,NNMF,50,100,189107,106373,200,False,9.959570,0.989600,False
8,Enzyme,3,LMF,50,100,189107,106373,100,None,2779.828726,1.160876,False
9,Enzyme,4,MSCMF,50,100,189107,106373,70,True,581.226073,6.516922,False


,family,outer_fold,method,protein_embedding_shape,compound_embedding_shape,expected_latent_dimension,shape_pass,finite_values_pass,nonnegative_pass,pass
0,Enzyme,1,MSCMF,"(664, 50)","(445, 50)",50,True,True,None,True
1,Enzyme,1,NNMF,"(664, 50)","(445, 50)",50,True,True,True,True
2,Enzyme,1,LMF,"(664, 50)","(445, 50)",50,True,True,None,True
3,Enzyme,2,MSCMF,"(664, 50)","(445, 50)",50,True,True,None,True
4,Enzyme,2,NNMF,"(664, 50)","(445, 50)",50,True,True,True,True
5,Enzyme,2,LMF,"(664, 50)","(445, 50)",50,True,True,None,True
6,Enzyme,3,MSCMF,"(664, 50)","(445, 50)",50,True,True,None,True
7,Enzyme,3,NNMF,"(664, 50)","(445, 50)",50,True,True,True,True
8,Enzyme,3,LMF,"(664, 50)","(445, 50)",50,True,True,None,True
9,Enzyme,4,MSCMF,"(664, 50)","(445, 50)",50,True,True,None,True


# Step 08 — Post-fit leakage and traceability audit

## 11. Verify saved outputs and role-to-latent traceability

The audit confirms that every saved embedding has the expected entity count and latent dimension, and that every pair in every role file can be reconstructed from valid protein and compound indices.

In [12]:
TRACEABILITY_ROWS = []

for family, cfg in FAMILIES.items():

    Y = RAW_DATA[family]["Y"]

    for outer_fold in range(
        1,
        N_OUTER_FOLDS + 1,
    ):

        entry = ROLE_DATA[
            family
        ][outer_fold]

        roles = entry["roles"]
        row_col = entry["row_col"]
        col_col = entry["col_col"]

        rows = roles[
            row_col
        ].astype(int).to_numpy()
        cols = roles[
            col_col
        ].astype(int).to_numpy()

        index_valid = (
            (rows >= 0).all()
            and (rows < Y.shape[0]).all()
            and (cols >= 0).all()
            and (cols < Y.shape[1]).all()
        )

        for method in METHODS:

            output_dir = fit_output_dir(
                family,
                outer_fold,
                method,
            )

            protein_file = (
                output_dir
                / "protein_embeddings.npy"
            )
            compound_file = (
                output_dir
                / "compound_embeddings.npy"
            )
            metadata_file = (
                output_dir
                / "fit_metadata.json"
            )

            files_exist = (
                protein_file.exists()
                and compound_file.exists()
                and metadata_file.exists()
            )

            assert files_exist

            A = np.load(
                protein_file,
                mmap_mode="r",
            )
            B = np.load(
                compound_file,
                mmap_mode="r",
            )

            all_pairs_reconstructable = (
                index_valid
                and rows.max() < A.shape[0]
                and cols.max() < B.shape[0]
            )

            assert all_pairs_reconstructable

            TRACEABILITY_ROWS.append({
                "family": cfg["display"],
                "outer_fold": outer_fold,
                "method": method,
                "saved_files_exist": (
                    files_exist
                ),
                "pair_indices_valid": (
                    index_valid
                ),
                "all_pairs_reconstructable": (
                    all_pairs_reconstructable
                ),
                "n_pairs_checked": (
                    len(roles)
                ),
                "pass": (
                    files_exist
                    and all_pairs_reconstructable
                ),
            })

TRACEABILITY_CHECK = pd.DataFrame(
    TRACEABILITY_ROWS
)

assert TRACEABILITY_CHECK["pass"].all()

display(TRACEABILITY_CHECK)

TRACEABILITY_CHECK.to_excel(
    STATS_DIR
    / "mf_traceability_check.xlsx",
    index=False,
    sheet_name="Traceability",
)

,family,outer_fold,method,saved_files_exist,pair_indices_valid,all_pairs_reconstructable,n_pairs_checked,pass
0,Enzyme,1,MSCMF,True,True,True,295480,True
1,Enzyme,1,NNMF,True,True,True,295480,True
2,Enzyme,1,LMF,True,True,True,295480,True
3,Enzyme,2,MSCMF,True,True,True,295480,True
4,Enzyme,2,NNMF,True,True,True,295480,True
5,Enzyme,2,LMF,True,True,True,295480,True
6,Enzyme,3,MSCMF,True,True,True,295480,True
7,Enzyme,3,NNMF,True,True,True,295480,True
8,Enzyme,3,LMF,True,True,True,295480,True
9,Enzyme,4,MSCMF,True,True,True,295480,True


## 12. Final leakage audit table

A fit is accepted only when:

- the train/validation/test role partition is disjoint;
- the mask includes all and only training entries;
- validation/test label perturbation leaves the masked supervised target unchanged;
- latent outputs have the expected shapes;
- latent outputs contain finite values;
- NNMF factors are non-negative;
- all pair indices can be mapped back to the saved latent factors.

In [13]:
FINAL_AUDIT_ROWS = []

for family, cfg in FAMILIES.items():

    for outer_fold in range(
        1,
        N_OUTER_FOLDS + 1,
    ):

        mask_row = MASK_AUDIT[
            (MASK_AUDIT["family"]
             == cfg["display"])
            & (MASK_AUDIT["outer_fold"]
               == outer_fold)
        ].iloc[0]

        role_row = ROLE_INTEGRITY_CHECK[
            (ROLE_INTEGRITY_CHECK["family"]
             == cfg["display"])
            & (
                ROLE_INTEGRITY_CHECK[
                    "outer_fold"
                ]
                == outer_fold
            )
        ].iloc[0]

        leakage_row = LEAKAGE_PRECHECK[
            (LEAKAGE_PRECHECK["family"]
             == cfg["display"])
            & (
                LEAKAGE_PRECHECK[
                    "outer_fold"
                ]
                == outer_fold
            )
        ].iloc[0]

        for method in METHODS:

            embedding_row = (
                EMBEDDING_AUDIT[
                    (
                        EMBEDDING_AUDIT[
                            "family"
                        ]
                        == cfg["display"]
                    )
                    & (
                        EMBEDDING_AUDIT[
                            "outer_fold"
                        ]
                        == outer_fold
                    )
                    & (
                        EMBEDDING_AUDIT[
                            "method"
                        ]
                        == method
                    )
                ].iloc[0]
            )

            trace_row = (
                TRACEABILITY_CHECK[
                    (
                        TRACEABILITY_CHECK[
                            "family"
                        ]
                        == cfg["display"]
                    )
                    & (
                        TRACEABILITY_CHECK[
                            "outer_fold"
                        ]
                        == outer_fold
                    )
                    & (
                        TRACEABILITY_CHECK[
                            "method"
                        ]
                        == method
                    )
                ].iloc[0]
            )

            mask_pass = (
                bool(
                    mask_row[
                        "train_weight_all_one"
                    ]
                )
                and bool(
                    mask_row[
                        "validation_weight_all_zero"
                    ]
                )
                and bool(
                    mask_row[
                        "test_weight_all_zero"
                    ]
                )
            )

            final_pass = (
                bool(role_row["pass"])
                and mask_pass
                and bool(
                    leakage_row["pass"]
                )
                and bool(
                    embedding_row["pass"]
                )
                and bool(trace_row["pass"])
            )

            FINAL_AUDIT_ROWS.append({
                "family": cfg["display"],
                "outer_fold": outer_fold,
                "method": method,
                "roles_disjoint": bool(
                    role_row["pass"]
                ),
                "train_only_mask": mask_pass,
                "heldout_label_perturbation_invariant": bool(
                    leakage_row["pass"]
                ),
                "embedding_audit_pass": bool(
                    embedding_row["pass"]
                ),
                "traceability_pass": bool(
                    trace_row["pass"]
                ),
                "final_pass": final_pass,
            })

FINAL_LEAKAGE_AUDIT = pd.DataFrame(
    FINAL_AUDIT_ROWS
)

assert FINAL_LEAKAGE_AUDIT[
    "final_pass"
].all(), (
    "At least one family/fold/method "
    "failed the final Step 08 audit."
)

display(FINAL_LEAKAGE_AUDIT)

FINAL_LEAKAGE_AUDIT.to_excel(
    STATS_DIR
    / "mf_final_leakage_audit.xlsx",
    index=False,
    sheet_name="Final Leakage Audit",
)

,family,outer_fold,method,roles_disjoint,train_only_mask,heldout_label_perturbation_invariant,embedding_audit_pass,traceability_pass,final_pass
0,Enzyme,1,MSCMF,True,True,True,True,True,True
1,Enzyme,1,NNMF,True,True,True,True,True,True
2,Enzyme,1,LMF,True,True,True,True,True,True
3,Enzyme,2,MSCMF,True,True,True,True,True,True
4,Enzyme,2,NNMF,True,True,True,True,True,True
5,Enzyme,2,LMF,True,True,True,True,True,True
6,Enzyme,3,MSCMF,True,True,True,True,True,True
7,Enzyme,3,NNMF,True,True,True,True,True,True
8,Enzyme,3,LMF,True,True,True,True,True,True
9,Enzyme,4,MSCMF,True,True,True,True,True,True


## 13. Protocol table



In [14]:
PROTOCOL_ROWS = [
    {
        "method": "MSCMF",
        "parameter": "latent_dimension",
        "value": MSCMF_K,
        "source_status": "explicit in supplied code",
    },
    {
        "method": "MSCMF",
        "parameter": "gamma",
        "value": MSCMF_GAMMA,
        "source_status": "mapped from supplied lambda_l value",
    },
    {
        "method": "MSCMF",
        "parameter": "lambda_protein",
        "value": MSCMF_LAMBDA_PROTEIN,
        "source_status": "mapped from supplied protein-similarity regularization value",
    },
    {
        "method": "MSCMF",
        "parameter": "lambda_compound",
        "value": MSCMF_LAMBDA_COMPOUND,
        "source_status": "mapped from supplied compound-similarity regularization value",
    },
    {
        "method": "MSCMF",
        "parameter": "similarity_regularization",
        "value": "graph Laplacian",
        "source_status": "controlled benchmark formulation",
    },
    {
        "method": "MSCMF",
        "parameter": "tolerance",
        "value": MSCMF_TOL,
        "source_status": "explicit in supplied code",
    },
    {
        "method": "MSCMF",
        "parameter": "max_iterations",
        "value": MSCMF_MAX_ITER,
        "source_status": "explicit in supplied code",
    },
    {
        "method": "NNMF",
        "parameter": "latent_dimension",
        "value": NNMF_K,
        "source_status": "standardized to 50 for controlled benchmark",
    },
    {
        "method": "NNMF",
        "parameter": "initialization",
        "value": "random",
        "source_status": "explicit in supplied code",
    },
    {
        "method": "NNMF",
        "parameter": "random_state",
        "value": NNMF_RANDOM_STATE,
        "source_status": "explicit in supplied code",
    },
    {
        "method": "NNMF",
        "parameter": "max_iterations",
        "value": NNMF_MAX_ITER,
        "source_status": (
            "made explicit for masked implementation"
        ),
    },
    {
        "method": "NNMF",
        "parameter": "tolerance",
        "value": NNMF_TOL,
        "source_status": (
            "made explicit for masked implementation"
        ),
    },
    {
        "method": "NNMF",
        "parameter": "objective",
        "value": "masked non-negative reconstruction",
        "source_status": "Step 07 leakage-safe adaptation",
    },
    {
        "method": "LMF",
        "parameter": "latent_dimension",
        "value": LMF_K,
        "source_status": "explicit in supplied code",
    },
    {
        "method": "LMF",
        "parameter": "learning_rate",
        "value": LMF_LEARNING_RATE,
        "source_status": "explicit in supplied code",
    },
    {
        "method": "LMF",
        "parameter": "max_iterations",
        "value": LMF_MAX_ITER,
        "source_status": "explicit in supplied code",
    },
    {
        "method": "LMF",
        "parameter": "l2_regularization",
        "value": False,
        "source_status": "not present in supplied implementation",
    },
    {
        "method": "LMF",
        "parameter": "loss_aggregation",
        "value": "sum",
        "source_status": "consistent with supplied unnormalised gradient",
    },
    {
        "method": "ALL",
        "parameter": "supervised_entries",
        "value": "train only",
        "source_status": "Step 07 leakage-safe adaptation",
    },
    {
        "method": "ALL",
        "parameter": "validation_mask_weight",
        "value": 0,
        "source_status": "Step 07 leakage-safe adaptation",
    },
    {
        "method": "ALL",
        "parameter": "test_mask_weight",
        "value": 0,
        "source_status": "Step 07 leakage-safe adaptation",
    },
    {
        "method": "ALL",
        "parameter": "heldout_labels_replaced_by_zero",
        "value": False,
        "source_status": "Step 07 leakage-safe adaptation",
    },
    {
        "method": "ALL",
        "parameter": "mf_fit_frequency",
        "value": "once per family x outer fold x method",
        "source_status": "controlled benchmark protocol",
    },
    {
        "method": "ALL",
        "parameter": "implementation_version",
        "value": IMPLEMENTATION_VERSION,
        "source_status": "resume-safety identifier",
    },
]

MF_PROTOCOL = pd.DataFrame(
    PROTOCOL_ROWS
)

display(MF_PROTOCOL)

MF_PROTOCOL.to_excel(
    STATS_DIR / "mf_protocol.xlsx",
    index=False,
    sheet_name="Protocol",
)

,method,parameter,value,source_status
0,MSCMF,latent_dimension,50,explicit in supplied code
1,MSCMF,gamma,0.1,mapped from supplied lambda_l value
2,MSCMF,lambda_protein,0.1,mapped from supplied protein-similarity regula...
3,MSCMF,lambda_compound,0.01,mapped from supplied compound-similarity regul...
4,MSCMF,similarity_regularization,graph Laplacian,controlled benchmark formulation
5,MSCMF,tolerance,0.001,explicit in supplied code
6,MSCMF,max_iterations,1000,explicit in supplied code
7,NNMF,latent_dimension,50,standardized to 50 for controlled benchmark
8,NNMF,initialization,random,explicit in supplied code
9,NNMF,random_state,0,explicit in supplied code


## 14. Save consolidated Step 07-08 statistics workbook



In [15]:
consolidated_file = (
    STATS_DIR
    / "mf_7_8_statistics.xlsx"
)

with pd.ExcelWriter(
    consolidated_file,
    engine="openpyxl",
) as writer:

    INPUT_AUDIT.to_excel(
        writer,
        sheet_name="Input Audit",
        index=False,
    )

    MASK_AUDIT.to_excel(
        writer,
        sheet_name="Mask Audit",
        index=False,
    )

    ROLE_INTEGRITY_CHECK.to_excel(
        writer,
        sheet_name="Role Integrity",
        index=False,
    )

    LEAKAGE_PRECHECK.to_excel(
        writer,
        sheet_name="Leakage Precheck",
        index=False,
    )

    FIT_STATS.to_excel(
        writer,
        sheet_name="Fit Statistics",
        index=False,
    )

    EMBEDDING_AUDIT.to_excel(
        writer,
        sheet_name="Embedding Audit",
        index=False,
    )

    TRACEABILITY_CHECK.to_excel(
        writer,
        sheet_name="Traceability",
        index=False,
    )

    FINAL_LEAKAGE_AUDIT.to_excel(
        writer,
        sheet_name="Final Leakage Audit",
        index=False,
    )

    MF_PROTOCOL.to_excel(
        writer,
        sheet_name="Protocol",
        index=False,
    )

print(
    "Consolidated statistics saved to:",
    consolidated_file
)

Consolidated statistics saved to: c:\Users\riskf\OneDrive\A-DTI2026\Stats\mf_7_8_statistics.xlsx


## 15. Save Step 07-08 manifest

In [16]:
manifest = {
    "implementation_version": IMPLEMENTATION_VERSION,
    "steps": [
        "07 - fit matrix factorization with validation/test labels masked",
        "08 - audit representation-learning leakage",
    ],
    "families": list(FAMILIES.keys()),
    "outer_folds": N_OUTER_FOLDS,
    "methods": METHODS,
    "parameters": {
        method: method_parameters(method)
        for method in METHODS
    },
    "masking_rule": {
        "train": 1,
        "validation": 0,
        "test": 0,
        "heldout_labels_replaced_by_zero": False,
    },
    "output_structure": (
        "data/split/<family>/latent/"
        "outer_fold_<k>/<method>/"
    ),
    "stats_workbook": str(
        consolidated_file.relative_to(
            PROJECT_ROOT
        )
    ),
}

manifest_file = (
    SPLIT_DIR
    / "mf_7_8_manifest.json"
)

manifest_file.write_text(
    json.dumps(
        manifest,
        indent=2,
    ),
    encoding="utf-8",
)

print(
    "Manifest saved to:",
    manifest_file
)

Manifest saved to: c:\Users\riskf\OneDrive\A-DTI2026\data\split\mf_7_8_manifest.json


## 16. Final integrity check



In [17]:
required_stats_files = [
    "mf_input_audit.xlsx",
    "mf_mask_audit.xlsx",
    "mf_role_integrity_check.xlsx",
    "mf_leakage_precheck.xlsx",
    "mf_fit_statistics.xlsx",
    "mf_embedding_audit.xlsx",
    "mf_traceability_check.xlsx",
    "mf_final_leakage_audit.xlsx",
    "mf_protocol.xlsx",
    "mf_7_8_statistics.xlsx",
]

for filename in required_stats_files:
    assert (
        STATS_DIR / filename
    ).exists(), (
        f"Missing Stats file: {filename}"
    )

assert len(FIT_STATS) == (
    len(FAMILIES)
    * N_OUTER_FOLDS
    * len(METHODS)
)

assert len(FINAL_LEAKAGE_AUDIT) == (
    len(FAMILIES)
    * N_OUTER_FOLDS
    * len(METHODS)
)

assert FINAL_LEAKAGE_AUDIT[
    "final_pass"
].all()

for family, cfg in FAMILIES.items():

    for outer_fold in range(
        1,
        N_OUTER_FOLDS + 1,
    ):

        for method in METHODS:

            output_dir = fit_output_dir(
                family,
                outer_fold,
                method,
            )

            assert (
                output_dir
                / "protein_embeddings.npy"
            ).exists()

            assert (
                output_dir
                / "compound_embeddings.npy"
            ).exists()

            assert (
                output_dir
                / "fit_metadata.json"
            ).exists()

    print(
        f"PASS - {cfg['display']}"
    )

assert manifest_file.exists()

print(
    "\nNotebook 7-8 completed successfully."
)
print(
    "Next: Step 09 - augmentation/resampling "
    "of inner-training data only."
)

PASS - Enzyme
PASS - GPCR
PASS - Ion Channel
PASS - Nuclear Receptor

Notebook 7-8 completed successfully.
Next: Step 09 - augmentation/resampling of inner-training data only.
